# Ingesta de Datos - Conteos Censales de Participación Ciudadana

Descarga, extracción y consolidación de los datasets de la plataforma CCPC del INE (2009-2024).

## 1. Librerias

In [ ]:
import gc
import zipfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from pathlib import Path
from tqdm import tqdm

## 2. Rutas del proyecto

In [ ]:
ROOT = Path().resolve().parent
DIR_ZIPS = ROOT / "DATA" / "RAW" / "ZIPS"
DIR_RAW = ROOT / "DATA" / "RAW"
DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"
DIR_DOCS = ROOT / "DATA" / "DOCS"

print(f"Raiz del proyecto: {ROOT}")
print(f"ZIPs:              {DIR_ZIPS}")
print(f"Datos crudos:      {DIR_RAW}")
print(f"Preprocesados:     {DIR_PREPROCESSED}")
print(f"Documentos:        {DIR_DOCS}")

## 3. Creación de directorios

In [ ]:
subdirs = [
    "conteos2009",
    "conteos2012",
    "conteos2015",
    "conteos2018",
    "conteos2021",
    "conteos2024",
    "conteosPJ2025",
    "voto_anticipado_2024",
    "prision_preventiva_2024",
]

DIR_ZIPS.mkdir(parents=True, exist_ok=True)
DIR_PREPROCESSED.mkdir(parents=True, exist_ok=True)
DIR_DOCS.mkdir(parents=True, exist_ok=True)
for d in subdirs:
    (DIR_RAW / d).mkdir(parents=True, exist_ok=True)

print("Directorios listos.")

## 4. URLs de descarga

Todas las fuentes de datos abiertos del INE para los Conteos Censales de Participación Ciudadana.

In [ ]:
DATASETS = {
    "conteos2009": "https://ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2009.zip",
    "conteos2012": "https://ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2012.zip",
    "conteos2015": "https://ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2015.zip",
    "conteos2018": "https://ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2018.zip",
    "conteos2021": "https://ine.mx/wp-content/uploads/2022/04/DatosAbiertos_DECEyEC_ConteosCensales2021.zip",
    "conteos2024": "https://ine.mx/wp-content/uploads/2025/08/DatosAbiertos_DECEyEC_ConteosCensales2024.zip",
    "conteosPJ2025": "https://ine.mx/wp-content/uploads/2026/09/DatosAbiertos_DECEyEC_ConteosCensalesPJ2025.zip",
    "voto_anticipado_2024": "https://ine.mx/wp-content/uploads/2025/08/DatosAbiertos_DECEyEC_ConteosCensales_VotoAnticipado2024.zip",
    "prision_preventiva_2024": "https://ine.mx/wp-content/uploads/2025/08/DatosAbiertos_DECEyEC_ConteosCensales_PrisionPreventiva2024.zip",
}

DOCUMENTOS = {
    "Convocatoria_Hackaton.pdf": "https://ine.mx/wp-content/uploads/2026/09/Convocatoria_Hackaton-1.pdf",
    "Carta_declaratoria_decir_verdad.pdf": "https://ine.mx/wp-content/uploads/2026/09/Carta-declaratoria-de-decir-la-verdad_Hackaton.pdf",
    "Carta_cesion_derechos.pdf": "https://ine.mx/wp-content/uploads/2026/09/Carta-de-cesion-de-derechos_Hackaton-1.pdf",
    "Manual_Conteos_Censales_2009_2024.pdf": "https://ine.mx/wp-content/uploads/2025/06/Manual-Conteos-Censales-2009-2024.pdf",
}

print(f"Datasets: {len(DATASETS)}")
for nombre, url in DATASETS.items():
    print(f"  {nombre}: {url.split('/')[-1]}")
print(f"\nDocumentos: {len(DOCUMENTOS)}")

## 5. Descarga de ZIPs y documentos

Descarga cada archivo si no existe ya en disco.

In [ ]:
def descargar(url, destino):
    if destino.exists():
        print(f"  Ya existe: {destino.name}")
        return
    print(f"  Descargando {destino.name}...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(destino, "wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))

print("=== Datasets (ZIPs) ===")
for nombre, url in DATASETS.items():
    archivo = url.split("/")[-1]
    descargar(url, DIR_ZIPS / archivo)

print("\n=== Documentos (PDFs) ===")
for nombre_local, url in DOCUMENTOS.items():
    descargar(url, DIR_DOCS / nombre_local)

## 6. Extracción de ZIPs

Cada ZIP se extrae en su propia carpeta dentro de `DATA/RAW/`.

In [ ]:
for nombre, url in DATASETS.items():
    archivo = url.split("/")[-1]
    zip_path = DIR_ZIPS / archivo
    destino = DIR_RAW / nombre
    csvs_existentes = list(destino.rglob("*.csv"))
    if csvs_existentes:
        print(f"  {nombre}: ya extraido ({len(csvs_existentes)} CSVs)")
        continue
    print(f"  Extrayendo {zip_path.name} -> DATA/RAW/{nombre}/")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(destino)
    csvs = list(destino.rglob("*.csv"))
    print(f"  {len(csvs)} archivos CSV extraidos.")

## 7. Inventario de archivos extraidos

In [ ]:
for nombre in DATASETS:
    carpeta = DIR_RAW / nombre
    csvs = sorted(carpeta.rglob("*.csv"))
    txts = sorted(carpeta.rglob("*.txt"))
    total_mb = sum(f.stat().st_size for f in csvs) / 1024 / 1024
    print(f"{nombre}:")
    print(f"  CSVs: {len(csvs)}  |  Diccionarios: {len(txts)}  |  Tamano: {total_mb:.1f} MB")

## 8. Carga de muestra y validación de esquema

Verificamos las columnas y tipos de cada dataset con un solo CSV de muestra.

In [ ]:
def cargar_muestra(nombre):
    carpeta = DIR_RAW / nombre
    csv = sorted(carpeta.rglob("*.csv"))[0]
    df = pd.read_csv(csv, encoding="utf-8-sig", nrows=5)
    print(f"--- {nombre} ({csv.name}) ---")
    print(f"Columnas: {list(df.columns)}")
    print(df.dtypes)
    print()
    return df

muestras = {}
for nombre in DATASETS:
    muestras[nombre] = cargar_muestra(nombre)

## 9. Consolidación del dataset federal (2009-2024)

Cada ZIP contiene un solo año. Concatenamos los 6 datasets de elecciones federales
y escribimos directamente a Parquet con PyArrow (un row group por año) para evitar
cargar los ~55M de filas en memoria simultáneamente.

In [ ]:
def consolidar_csvs(nombre):
    carpeta = DIR_RAW / nombre
    csvs = sorted(carpeta.rglob("*.csv"))
    partes = []
    for csv in tqdm(csvs, desc=nombre):
        df = pd.read_csv(csv, encoding="utf-8-sig", dtype={"DEF": str, "DEL": str})
        partes.append(df)
    return pd.concat(partes, ignore_index=True)

FEDERALES = ["conteos2009", "conteos2012", "conteos2015",
             "conteos2018", "conteos2021", "conteos2024"]

SCHEMA = pa.schema([
    ("AELEC", pa.int64()), ("FELECCION", pa.string()),
    ("EDOCVE", pa.int64()), ("EDONOM", pa.string()),
    ("MPIOCVE", pa.int64()), ("MPIONOM", pa.string()),
    ("SECCION", pa.int64()), ("TIPOSEC", pa.string()),
    ("DEF", pa.string()), ("DEL", pa.string()),
    ("SEXO", pa.int64()), ("EDAD", pa.int64()),
    ("LN", pa.int64()), ("SV", pa.int64()),
    ("NV", pa.int64()), ("NS", pa.int64()),
])

out_federal = DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet"
writer = pq.ParquetWriter(out_federal, SCHEMA)
total_rows = 0

for nombre in FEDERALES:
    df_year = consolidar_csvs(nombre)
    n = len(df_year)
    total_rows += n
    print(f"  {nombre}: {n:,} filas")
    table = pa.Table.from_pandas(df_year, schema=SCHEMA, preserve_index=False)
    writer.write_table(table)
    del df_year, table
    gc.collect()

writer.close()
print(f"\nTotal consolidado: {total_rows:,} filas")
print(f"Archivo: {out_federal.name} ({out_federal.stat().st_size / 1024**2:.1f} MB)")

## 10. Resumen del dataset federal

In [ ]:
pf = pq.ParquetFile(DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet")
print(f"Total filas: {pf.metadata.num_rows:,}")
print(f"Row groups: {pf.metadata.num_row_groups}")
print(f"Columnas: {pf.schema_arrow.names}")

df_sample = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["AELEC", "EDOCVE", "SECCION"]
)
print(f"\nAnios de eleccion: {sorted(df_sample['AELEC'].unique())}")
print(f"Entidades: {df_sample['EDOCVE'].nunique()}")
print(f"Secciones: {df_sample['SECCION'].nunique()}")
print(f"\nRegistros por anio:")
print(df_sample.groupby("AELEC").size().to_string())
del df_sample

## 11. Esquema del parquet federal

Los tipos se definieron al escribir el parquet con un schema de PyArrow explicito.

In [ ]:
pf = pq.ParquetFile(DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet")
for i in range(len(pf.schema_arrow)):
    field = pf.schema_arrow.field(i)
    print(f"  {field.name:12s}  {field.type}")
print(f"\nTamano en disco: {out_federal.stat().st_size / 1024**2:.1f} MB")

## 12. Consolidación del dataset PJ 2025

In [ ]:
carpeta_pj = DIR_RAW / "conteosPJ2025"
csvs_pj = sorted(carpeta_pj.rglob("*.csv"))
partes_pj = []
for csv in tqdm(csvs_pj, desc="conteosPJ2025"):
    partes_pj.append(pd.read_csv(csv, encoding="utf-8-sig", dtype={"DEF": str, "CJ": str, "DEJ": str}))
df_pj2025 = pd.concat(partes_pj, ignore_index=True)
del partes_pj
print(f"Filas: {len(df_pj2025):,}")
print(f"Columnas: {list(df_pj2025.columns)}")

## 13. Consolidación de datasets complementarios

Voto anticipado y prisión preventiva (solo 2024).

In [ ]:
extras = ["voto_anticipado_2024", "prision_preventiva_2024"]
df_extras = {}

for nombre in extras:
    carpeta = DIR_RAW / nombre
    csvs = sorted(carpeta.rglob("*.csv"))
    partes = [pd.read_csv(c, encoding="utf-8-sig") for c in csvs]
    df = pd.concat(partes, ignore_index=True)
    df_extras[nombre] = df
    print(f"{nombre}: {len(df):,} filas | Columnas: {list(df.columns)}")

## 14. Revisión de valores nulos y no especificados

In [ ]:
df_ns = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["NS"]
)
total = len(df_ns)
ns_pos = (df_ns["NS"] > 0).sum()
print(f"Total registros: {total:,}")
print(f"Registros con NS > 0: {ns_pos:,}")
print(f"Porcentaje NS > 0: {ns_pos / total * 100:.2f}%")
del df_ns

## 15. Guardado de datasets complementarios

El parquet federal ya se guardo en el paso 9. Aquí guardamos PJ 2025 y extras.

In [ ]:
df_pj2025.to_parquet(DIR_PREPROCESSED / "conteos_pj2025.parquet", index=False)
print(f"Guardado: conteos_pj2025.parquet ({len(df_pj2025):,} filas)")
del df_pj2025

for nombre, df in df_extras.items():
    fname = f"{nombre}.parquet"
    df.to_parquet(DIR_PREPROCESSED / fname, index=False)
    print(f"Guardado: {fname} ({len(df):,} filas)")
del df_extras
gc.collect()

print("\nArchivos en DATA/PREPROCESSED/:")
for f in sorted(DIR_PREPROCESSED.glob("*.parquet")):
    print(f"  {f.name}: {f.stat().st_size / 1024**2:.1f} MB")

## 16. Verificación de lectura desde Parquet

In [ ]:
pf = pq.ParquetFile(DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet")
print(f"Lectura OK: {pf.metadata.num_rows:,} filas x {len(pf.schema_arrow.names)} columnas")

df_check = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["AELEC"]
)
print(f"Anios: {sorted(df_check['AELEC'].unique())}")
del df_check

df_head = pf.read_row_group(0).to_pandas().head(3)
df_head

## Resumen

Datasets consolidados en `DATA/PREPROCESSED/`:

| Archivo | Contenido |
|---|---|
| `conteos_federal_2009_2024.parquet` | 6 elecciones federales concatenadas (2009, 2012, 2015, 2018, 2021, 2024) |
| `conteos_pj2025.parquet` | Elección Poder Judicial 2025 |
| `voto_anticipado_2024.parquet` | Voto anticipado 2024 |
| `prision_preventiva_2024.parquet` | Voto en prisión preventiva 2024 |